Завдання: Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних;

In [1]:
import urllib.request
import datetime
import os

folder_name = "vhi_data"
os.makedirs(folder_name, exist_ok=True)

def download_noaa_data():
    
    current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    for province_id in range(1, 28):
        existing_files = [f for f in os.listdir(folder_name) if f.startswith(f"vhi_id_{province_id}_")]
        
        if len(existing_files) > 0:
            print(f"Дані  {province_id} вже існують ({existing_files[0]})")
            continue
            
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2026&type=Mean"

        filename = f"{folder_name}/vhi_id_{province_id}_{current_time}.csv"
        
        try:
            urllib.request.urlretrieve(url, filename)
            print(f"Успіх файл: {filename}")
        except Exception as e:
            print(f"Помилка для {province_id}: {e}")

    print("Процедура завершена")
download_noaa_data()

Успіх файл: vhi_data/vhi_id_1_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_2_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_3_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_4_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_5_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_6_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_7_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_8_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_9_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_10_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_11_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_12_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_13_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_14_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_15_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_16_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_17_2026-03-21_01-46-42.csv
Успіх файл: vhi_data/vhi_id_18_2026-03-21_01-46-42.csv
Успіх файл: vhi_dat

Завдання зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning.

In [2]:
import pandas as pd
import os

folder_name = "vhi_data"
headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
frames = []

for filename in os.listdir(folder_name):
    if filename.endswith(".csv"):
        filepath = os.path.join(folder_name, filename)
        df = pd.read_csv(filepath, header=1, names=headers)
        df = df.drop(columns=['empty'], errors='ignore')
        
        df['Year'] = df['Year'].astype(str).str.replace('<tt>', '').str.replace('</pre>', '')
        df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
        
        df = df.dropna()
        df['Year'] = df['Year'].astype(int)
        
        df = df[df['VHI'] != -1]
        
        old_id = int(filename.split('_')[2])
        df['NOAA_ID'] = old_id
        
        frames.append(df)

vhi_df = pd.concat(frames, ignore_index=True)

print("Дані зчитано та очищено. Перші 5 рядків:")
display(vhi_df.head())

Дані зчитано та очищено. Перші 5 рядків:


,Year,Week,SMN,SMT,VCI,TCI,VHI,NOAA_ID
0,1982,2.0,0.063,261.53,55.89,38.20,47.04,10
1,1982,3.0,0.063,263.45,57.30,32.69,44.99,10
2,1982,4.0,0.061,265.10,53.96,28.62,41.29,10
3,1982,5.0,0.058,266.42,46.87,28.57,37.72,10
4,1982,6.0,0.056,267.47,39.55,30.27,34.91,10


Додати стовпчики з назвою та індексом області

In [3]:
vhi_df['Area_ID'] = vhi_df['NOAA_ID']
vhi_df['Area_Name'] = None 

print("Стовпчики Area_ID та Area_Name  додано:")
display(vhi_df.head())

Стовпчики Area_ID та Area_Name  додано:


,Year,Week,SMN,SMT,VCI,TCI,VHI,NOAA_ID,Area_ID,Area_Name
0,1982,2.0,0.063,261.53,55.89,38.20,47.04,10,10,None
1,1982,3.0,0.063,263.45,57.30,32.69,44.99,10,10,None
2,1982,4.0,0.061,265.10,53.96,28.62,41.29,10,10,None
3,1982,5.0,0.058,266.42,46.87,28.57,37.72,10,10,None
4,1982,6.0,0.056,267.47,39.55,30.27,34.91,10,10,None


Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 

In [4]:
index_mapping = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21, 
    11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 
    20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
}

names_mapping = {
    1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька", 5: "Житомирська",
    6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська", 9: "Київська", 10: "Кіровоградська",
    11: "Луганська", 12: "Львівська", 13: "Миколаївська", 14: "Одеська", 15: "Полтавська",
    16: "Рівненська", 17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
    21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська", 25: "Республіка Крим",
    26: "м. Київ", 27: "м. Севастополь"
}

vhi_df['Area_ID'] = vhi_df['NOAA_ID'].map(index_mapping)
vhi_df['Area_Name'] = vhi_df['Area_ID'].map(names_mapping)

vhi_df = vhi_df.drop(columns=['NOAA_ID'])

print("Індекси змінено")
display(vhi_df.head())

Індекси змінено


,Year,Week,SMN,SMT,VCI,TCI,VHI,Area_ID,Area_Name
0,1982,2.0,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
1,1982,3.0,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
2,1982,4.0,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
3,1982,5.0,0.058,266.42,46.87,28.57,37.72,21,Хмельницька
4,1982,6.0,0.056,267.47,39.55,30.27,34.91,21,Хмельницька


Реалізувати процедуру для формування вибірки: Ряд VHI для області за вказаний рік.

In [5]:
def get_vhi_for_year(df, province_id, year):
    result = df[(df['Area_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]
    return result

print("Ряд VHI для Вінницької області за 2020 рік:")
display(get_vhi_for_year(vhi_df, province_id=1, year=2020).head(10))

Ряд VHI для Вінницької області за 2020 рік:


,Week,VHI
35645,1.0,40.92
35646,2.0,43.19
35647,3.0,44.74
35648,4.0,45.29
35649,5.0,44.80
35650,6.0,43.92
35651,7.0,43.10
35652,8.0,42.88
35653,9.0,43.71
35654,10.0,45.61


Реалізувати процедуру для формування вибірки: Ряд VHI за вказаний діапазон років для вказаних областей.

In [6]:
def get_vhi_for_years_and_provinces(df, province_ids, start_year, end_year):
    result = df[(df['Area_ID'].isin(province_ids)) & (df['Year'] >= start_year) & 
        (df['Year'] <= end_year)][['Year', 'Week', 'Area_ID', 'Area_Name', 'VHI']]
    return result

print("Рряд VHI для Вінницької та Волинської областей за період 2018-2020:")
display(get_vhi_for_years_and_provinces(vhi_df, province_ids=[1, 2], start_year=2018, end_year=2020).head(10))

Рряд VHI для Вінницької та Волинської областей за період 2018-2020:


,Year,Week,Area_ID,Area_Name,VHI
35541,2018,1.0,1,Вінницька,46.61
35542,2018,2.0,1,Вінницька,49.50
35543,2018,3.0,1,Вінницька,52.71
35544,2018,4.0,1,Вінницька,53.30
35545,2018,5.0,1,Вінницька,51.76
35546,2018,6.0,1,Вінницька,49.91
35547,2018,7.0,1,Вінницька,47.45
35548,2018,8.0,1,Вінницька,46.65
35549,2018,9.0,1,Вінницька,48.45
35550,2018,10.0,1,Вінницька,49.56


Пошук екстремумів для вказаних областей та років, середнього, медіани.

In [7]:
def get_vhi_statistics(df, province_ids, years):
    filtered_df = df[(df['Area_ID'].isin(province_ids)) & (df['Year'].isin(years))]
    
    stats = {
        'Мінімум': filtered_df['VHI'].min(),
        'Максимум': filtered_df['VHI'].max(),
        'Середнє': round(filtered_df['VHI'].mean(), 2),
        'Медіана': filtered_df['VHI'].median()
    }
    
    return pd.DataFrame([stats])

print("Статистика VHI (min, max, mean, median) для Вінницької та Волинської областей за 2019-2020 роки:")
display(get_vhi_statistics(vhi_df, province_ids=[1, 2], years=[2019, 2020]))

Статистика VHI (min, max, mean, median) для Вінницької та Волинської областей за 2019-2020 роки:


,Мінімум,Максимум,Середнє,Медіана
0,20.26,67.33,47.6,47.93
